# 1. Tổng quan

Mục tiêu là khám phá các thuật toán học máy giám sát để phát hiện hành vi dựa trên dữ liệu cảm biến. Chúng ta sẽ xử lý các vấn đề như giá trị thiếu, xây dựng đường ống (pipelines), tạo ra sơ đồ phụ thuộc, và áp dụng mã hóa một lần.

Chúng ta sẽ sử dụng dữ liệu từ các cảm biến để dự đoán hành vi của các đối tượng trong môi trường. Các thuật toán học máy sẽ giúp phân loại hành vi dựa trên các đặc điểm đo lường được từ dữ liệu cảm biến. Mục tiêu cuối cùng là xây dựng một mô hình có khả năng dự đoán chính xác hành vi dựa trên các tính năng cảm biến được cung cấp.

# Outline

## 1.1 Tải và Khám Phá Dữ Liệu

Tải dữ liệu và tiến hành kiểm tra sơ bộ (kích thước, loại dữ liệu, và các thuộc tính).

Phân tích thống kê cho các tính năng trong dataset, bao gồm:
* Giá trị thiếu và cách xử lý.
* Các tính năng đặc biệt như giới tính, tuổi, hành vi.
* Kiểm tra sự phân phối mục tiêu và đặc tính dữ liệu qua biểu đồ phân phối và biểu đồ hộp.

## 1.2 Tiền Xử Lý Dữ Liệu

* Xử lý các giá trị thiếu bằng các phương pháp phù hợp.
* Tách biệt dữ liệu thành các nhóm: cảm biến IMU, ToF, và mục tiêu.
* Chuẩn hóa và chuẩn bị dữ liệu cho mô hình.
* Feature Engineering để tối ưu hóa đặc trưng dữ liệu.

## 1.3 Khám Phá Dữ Liệu và Trực Quan Hóa

Trực quan hóa mối quan hệ giữa các tính năng:
* Tương quan Pearson giữa các tính năng.
* Biểu đồ phân tán giữa các tính năng chính (ví dụ: chiều cao, trọng lượng, v.v.).
* Biểu đồ hình tròn cho các tính năng phân loại như giới tính.
* Tính toán mối tương quan giữa các dữ liệu cảm biến (IMU, ToF, v.v.).

## 1.4 Xây Dựng Mô Hình Học Máy

Chọn mô hình học máy phù hợp: Linear Regression, Decision Tree, Random Forest, XGBoost, MLP, và các mô hình hồi quy khác.

Tối ưu hóa mô hình qua Grid Search và Cross-Validation (K-fold).

Tối ưu hóa siêu tham số để cải thiện độ chính xác.

## 1.5 Đánh Giá Mô Hình và Tối Ưu Hóa

Đánh giá mô hình dựa trên các chỉ số như F1-score, accuracy, precision, recall.

Sử dụng ensemble learning như bagging và boosting để kết hợp các mô hình và cải thiện hiệu suất.

Tối ưu hóa mô hình cuối cùng để đảm bảo tính ổn định và khả năng dự đoán chính xác trong các bộ dữ liệu thực tế.

## 1.6 Kết Quả và Thảo Luận


Trình bày kết quả mô hình với các chỉ số đánh giá chi tiết.

Phân tích các yếu tố ảnh hưởng đến độ chính xác của mô hình.

Kết luận về mô hình tối ưu và khả năng áp dụng vào thực tế.

# 2. Tổng quan dữ liệu

## 2.1 Tải dữ liệu và thư viện

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import warnings
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.mplot3d import Axes3D
import os
import polars as pl
from sklearn.metrics import accuracy_score, f1_score
import joblib
from scipy.spatial.transform import Rotation as R

ModuleNotFoundError: No module named 'kaggle_evaluation'

In [ ]:
# Tải tập dữ liệu
#Dữ liệu cảm biến cho tập huấn luyện (Train Sensor Data):
train_df = pd.read_csv("/kaggle/input/cmi-detect-behavior-with-sensor-data/train.csv")

#Thông tin nhân khẩu học của đối tượng trong tập huấn luyện (giới tính, tuổi, ...):
train_dem_df = pd.read_csv("/kaggle/input/cmi-detect-behavior-with-sensor-data/train_demographics.csv")

#Dữ liệu cảm biến cho tập kiểm tra:
test_df = pd.read_csv("/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv")

# Thông tin nhân khẩu học của đối tượng trong tập kiểm tra:
test_dem_df = pd.read_csv("/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv")

## 2.1 Khám phá dữ liệu

### 2.1.3 Tóm tắt thống kê các tính năng

In [ ]:
datasets = {
    "Train Data": train_df,
    "Train Demographics": train_dem_df,
    "Test Data": test_df,
    "Test Demographics": test_dem_df,
}

for name, df in datasets.items():
    print(f"Missing values in {name}:")
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        print(missing)
    else:
        print("  ✅ No missing values.")
    print()


### 2.1.5 kiểm tra giá trị trùng lặp

In [ ]:
#Kiểm tra trùng lặp (Duplicate Rows): .duplicated(): Trả về Series boolean, dòng nào trùng lặp sẽ là True.
#                                     .sum(): Đếm tổng số dòng trùng lặp.
#Kiểm tra xem dữ liệu có bản ghi nào bị trùng không — quan trọng cho việc làm sạch dữ liệu.

# Đếm các hàng trùng lặp trong train_df
train_duplicates = train_df.duplicated().sum()

# Đếm các hàng trùng lặp trong  test_df
test_duplicates = test_df.duplicated().sum()

# Đếm các hàng trùng lặp trong train_dem_df (optional)
train_dem_duplicates = train_dem_df.duplicated().sum()
# Đếm các hàng trùng lặp trong test_dem_df (optional)
test_dem_duplicates = test_dem_df.duplicated().sum()

# In số lượng dòng trùng:
print(f"Number of duplicate rows in train_df: {train_duplicates}")
print(f"Number of duplicate rows in test_df: {test_duplicates}")
print(f"Number of duplicate rows in train_dem_df: {train_dem_duplicates}")
print(f"Number of duplicate rows in test_dem_df: {test_dem_duplicates}")

## Tiền xử lý dữ liệu

### Tỷ lệ phần trăm Null

#### UTILS/HELPERS

In [ ]:
def null_percent(df):
    per=((df.isnull().sum()/len(df))*100).round(2)
    return per[per>0]

print("Nan Values in Train data")
print(null_percent(train_df))

Các cột có tỷ lệ NaN cao nhất là các cột có tên bắt đầu với tof_5_v (5.24% NaN).

Các cột khác như rot_x, rot_w có tỷ lệ NaN thấp hơn nhiều (0.64%).

## 2.2 Quan sát hoặc xu hướng ban đầu

In [ ]:
# --- 2.4 Hợp nhất dữ liệu nhân khẩu học và cảm biến ---
merged_train = pd.merge(train_df, train_dem_df, on="subject", how="left")
merged_test = pd.merge(test_df, test_dem_df, on="subject", how="left")

print("Merged Train Shape:", merged_train.shape)
print("Merged Test Shape:", merged_test.shape)
display(merged_train.head(2))


**Merged Train Shape:** (574945, 348) - Sau khi hợp nhất, merged_train có 574,945 dòng và 348 cột, tức là dữ liệu từ train_df (574,945 dòng và 341 cột) đã được kết hợp với 7 cột từ train_dem_df (vì train_dem_df có 8 cột, nhưng 1 cột là subject đã được sử dụng làm khóa hợp nhất).

**Merged Test Shape:** (107, 343) - Tương tự, merged_test có 107 dòng và 343 cột, kết quả là hợp nhất giữa test_df (107 dòng và 336 cột) với test_dem_df (8 cột, với 1 cột subject là khóa hợp nhất).

In [ ]:
# --- Thống kê mô tả toàn bộ merged_train ---
merged_train.describe().T


In [ ]:
#Kiểm tra thiếu giá trị & thống kê cảm biến:
#Danh sách các cột không thuộc cảm biến (có thể là ID, thông tin khác).

excluded_prefixes = ('acc_', 'rot_', 'thm_', 'tof_')
sensor_cols = [col for col in train_df.columns if not col.startswith(excluded_prefixes)]

# Sensor Data Summary for TRAIN
#isnull().sum(): Đếm số giá trị bị thiếu.
missing_sensor_train = pd.DataFrame({
    'Feature': sensor_cols,
    '[TRAIN] Missing Count': train_df[sensor_cols].isnull().sum().values,
    '[TRAIN] Missing %': (train_df[sensor_cols].isnull().sum().values / len(train_df)) * 100
})

#nunique(): Đếm số lượng giá trị duy nhất.
unique_sensor_train = pd.DataFrame({
    'Feature': sensor_cols,
    'Unique Values [TRAIN]': train_df[sensor_cols].nunique().values
})

#dtypes: Lấy kiểu dữ liệu của từng cột.
dtypes_sensor = pd.DataFrame({
    'Feature': sensor_cols,
    'Data Type': train_df[sensor_cols].dtypes.values
})

# Merge all summaries (NO test set)
#merge: Gộp các bảng thống kê thành bảng duy nhất theo Feature.
sensor_summary = missing_sensor_train \
    .merge(unique_sensor_train, on='Feature', how='left') \
    .merge(dtypes_sensor, on='Feature', how='left')

# Display styled DataFrame (mask NaNs just for styling)
#fillna(0): Điền giá trị thiếu bằng 0 (cho đẹp mắt khi hiển thị).
#.style.background_gradient: Tô màu nền theo giá trị giúp dễ nhìn.
styled_df = sensor_summary.fillna(0)
styled_df.style.background_gradient(cmap='viridis')

- Không có giá trị thiếu: Tất cả các cột trong bảng thống kê đều không có giá trị thiếu (Missing Count = 0), cho thấy dữ liệu của bạn rất đầy đủ.

- Sự đa dạng trong dữ liệu: Các cột như row_id, sequence_id, và subject có số lượng giá trị duy nhất khá lớn, trong khi các cột như sequence_type, phase, và behavior có ít giá trị duy nhất hơn, cho thấy chúng có thể là các thông tin phân loại.

- Kiểu dữ liệu: Hầu hết các cột đều có kiểu dữ liệu object, ngoại trừ sequence_counter có kiểu int64, điều này cho thấy hầu hết các cột chứa thông tin phân loại (string), chỉ có một cột chứa giá trị số nguyên.

In [ ]:
#Thống kê tương tự cho nhân khẩu học: Tương tự như phần thống kê cảm biến nhưng áp dụng cho dữ liệu nhân khẩu học.

# Cột nhân khẩu học (không loại trừ)
dem_cols = train_dem_df.columns

# Giá trị bị thiếu trong dữ liệu nhân khẩu học của train
missing_demo_train = pd.DataFrame({
    'Feature': dem_cols,
    '[TRAIN DEMO] Missing Count': train_dem_df[dem_cols].isnull().sum().values,
    '[TRAIN DEMO] Missing %': (train_dem_df[dem_cols].isnull().sum().values / len(train_dem_df)) * 100
})

# Giá trị duy nhất được tính trong dữ liệu nhân khẩu học của train
unique_demo_train = pd.DataFrame({
    'Feature': dem_cols,
    'Unique Values [TRAIN DEMO]': train_dem_df[dem_cols].nunique().values
})

# Data types
dtypes_demo = pd.DataFrame({
    'Feature': dem_cols,
    'Data Type': train_dem_df[dem_cols].dtypes.values
})

# Tóm tắt hợp nhất (chỉ dành cho đào tạo)
demo_summary = (
    missing_demo_train
    .merge(unique_demo_train, on='Feature', how='left')
    .merge(dtypes_demo, on='Feature', how='left')
)

# Hiển thị tóm tắt theo phong cách
demo_summary.style.background_gradient(cmap='viridis')

- Không có giá trị thiếu: Các cột nhân khẩu học đều không có giá trị thiếu, cho thấy dữ liệu đầy đủ và sẵn sàng để phân tích.

- Sự đa dạng trong dữ liệu: Các cột như subject, age, và height_cm có sự đa dạng về giá trị duy nhất, trong khi các cột như sex, handedness có ít giá trị duy nhất hơn, cho thấy đây là các cột phân loại.

- Kiểu dữ liệu: Các cột phần lớn có kiểu dữ liệu int64 hoặc float64, cho thấy dữ liệu chủ yếu là số nguyên hoặc số thực. Các cột như subject có kiểu dữ liệu object, có thể là các chuỗi ký tự.

### 2.2.2. Sensor-Based Features (Tính năng dựa trên cảm biến)

#### **NOTE:**
* acc_: Gia tốc kế.
* rot_: Cảm biến quay.
* thm_: Nhiệt độ, độ ẩm.
* tof_: Khoảng cách.


# 3. EDA

## 3.1. Numerical Feature Distribution Analysis (Phân tích phân phối tính năng số) (Phân tích đơn biến)

### 3.1.1 Participant & Contextual Features (Tính năng của người tham gia và bối cảnh)

### 3.1.2 Sensor-Based Features (Tính năng dựa trên cảm biến)

**Mục tiêu:**
Trích xuất các đặc trưng thống kê (mean, std, max) từ dữ liệu sensor (acceleration, rotation, temperature, ToF) theo từng sequence_id để phục vụ mô hình học máy hoặc phân tích dữ liệu.

In [ ]:
import numpy as np
import pandas as pd

# 1) Sao chép đào tạo và kiểm tra để chúng ta không sửa đổi DataFrame gốc
train_temp = train_df.copy()
test_temp  = test_df.copy()

# 2) GIA TỐC KẾ: tính toán độ lớn tại mỗi dấu thời gian
train_temp['acc_mag'] = np.sqrt(
    train_temp['acc_x']**2 + train_temp['acc_y']**2 + train_temp['acc_z']**2
)
test_temp['acc_mag'] = np.sqrt(
    test_temp['acc_x']**2 + test_temp['acc_y']**2 + test_temp['acc_z']**2
)

# 3) ROTATION: tính toán “góc quay” từ thành phần quaternion w
# (Lưu ý: rot_w nằm trong [-1,1], do đó arccos hợp lệ. Chúng tôi bỏ qua NaN nếu có.)
train_temp['rot_angle'] = 2 * np.arccos(train_temp['rot_w'].clip(-1,1))
test_temp['rot_angle']  = 2 * np.arccos(test_temp['rot_w'].clip(-1,1))

# 4) Nhóm theo sequence_id và tổng hợp các tóm tắt gia tốc kế
acc_agg_funcs = {
    'acc_mag': ['mean', 'std', 'max']
}
train_acc_summary = train_temp.groupby('sequence_id').agg(acc_agg_funcs)
test_acc_summary  = test_temp.groupby('sequence_id').agg(acc_agg_funcs)

# Làm phẳng cột MultiIndex
train_acc_summary.columns = ['acc_mag_' + stat for stat in ['mean', 'std', 'max']]
test_acc_summary.columns  = ['acc_mag_' + stat for stat in ['mean', 'std', 'max']]

# 5) Nhóm theo sequence_id và tổng hợp tóm tắt vòng quay
rot_agg_funcs = {
    'rot_angle': ['mean', 'std', 'max']
}
train_rot_summary = train_temp.groupby('sequence_id').agg(rot_agg_funcs)
test_rot_summary  = test_temp.groupby('sequence_id').agg(rot_agg_funcs)

train_rot_summary.columns = ['rot_angle_' + stat for stat in ['mean', 'std', 'max']]
test_rot_summary.columns  = ['rot_angle_' + stat for stat in ['mean', 'std', 'max']]

# 6) NHIỆT ĐỘ: năm cảm biến thm_1 … thm_5
thm_cols = [f"thm_{i}" for i in range(1, 6)]

# Xác định hàm tổng hợp: trung bình + độ lệch chuẩn
thm_agg_funcs = {col: ['mean', 'std'] for col in thm_cols}

train_thm_summary = train_temp.groupby('sequence_id').agg(thm_agg_funcs)
test_thm_summary  = test_temp.groupby('sequence_id').agg(thm_agg_funcs)

# Làm phẳng các cột MultiIndex
flattened_thm_cols = []
for sensor in thm_cols:
    for stat in ['mean','std']:
        flattened_thm_cols.append(f"{sensor}_{stat}")

train_thm_summary.columns = flattened_thm_cols
test_thm_summary.columns  = flattened_thm_cols

# 7) THỜI GIAN BAY: mỗi cảm biến i có 64 cột pixel: tof_i_v0 … tof_i_v63
# Chúng ta sẽ tạo một “tof_i_mean_at_ts” cho mỗi dấu thời gian, sau đó tổng hợp theo từng chuỗi.

def compute_tof_sequence_summary(df):
    # Khởi tạo một dict để giữ DataFrames theo từng chuỗi
    seq_summaries = {}

    for i in range(1, 6):
        # Xây dựng danh sách các cột cho cảm biến i
        tof_cols = [f"tof_{i}_v{pix}" for pix in range(64)]
        # Thay thế -1 bằng NaN để chúng không làm lệch giá trị trung bình; chuyển đổi sang float
        ts_grid = df[tof_cols].replace(-1, np.nan).astype(float)
        # Tính toán “trung bình trên tất cả 64 pixel” cho mỗi dấu thời gian
        df[f"tof_{i}_mean_at_ts"] = ts_grid.mean(axis=1)

   # Bây giờ, nhóm theo id chuỗi và tính giá trị trung bình và độ lệch chuẩn của các giá trị trung bình đó
    agg_dict = {f"tof_{i}_mean_at_ts": ['mean','std'] for i in range(1, 6)}
    summary = df.groupby('sequence_id').agg(agg_dict)
    # Làm phẳng các cột MultiIndex
    flat_cols = [f"tof_{i}_{stat}" for i in range(1, 6) for stat in ['mean','std']]
    summary.columns = flat_cols
    return summary

train_tof_summary = compute_tof_sequence_summary(train_temp)
test_tof_summary  = compute_tof_sequence_summary(test_temp)

# 8) Hợp nhất các tóm tắt accel, rotation, thm, tof (trên sequence_id)
train_sensor_summary = (
    train_acc_summary
    .join(train_rot_summary, how='outer')
    .join(train_thm_summary, how='outer')
    .join(train_tof_summary, how='outer')
)

test_sensor_summary = (
    test_acc_summary
    .join(test_rot_summary, how='outer')
    .join(test_thm_summary, how='outer')
    .join(test_tof_summary, how='outer')
)

# 9) Thêm cột “Dataset” để chúng ta có thể thực hiện box+hist song song
train_sensor_summary['Dataset'] = 'Train'
test_sensor_summary['Dataset']  = 'Test'

# 10) Ghép nối thành một DataFrame để vẽ đồ thị
combined_sensor_summary = pd.concat(
    [train_sensor_summary, test_sensor_summary],
    axis=0
).reset_index(drop=True)

## 3.3 Target Feature Analysis (Univariate Analysis)

### 3.4.2 Sensor‐Based Differentiation Across Demographics and Physiology Features

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# (1) Sáp nhập train_demographics vào train_df nếu chưa thực hiện
train_df = train_df.merge(
    train_dem_df,
    on="subject",
    how="left"
)

## Xử lý missing values

### 1. IMU – Gia tốc/quaternion (acc_*, rot_w/x/y/z)

In [ ]:
def fix_imu(df):
    imu_cols = [c for c in df.columns if c.startswith(("acc_", "rot_"))]
    def _seq_fix(g):
        g[imu_cols] = g[imu_cols].interpolate(limit_direction="both")
        # renormalize quaternion
        q = g[["rot_w","rot_x","rot_y","rot_z"]].to_numpy(float)
        n = (q**2).sum(1)**0.5
        n[n==0] = 1.0
        q = q / n[:,None]
        g[["rot_w","rot_x","rot_y","rot_z"]] = q
        return g
    return df.groupby("sequence_id", group_keys=False).apply(_seq_fix)


### 2. Thermopile (thm_1..thm_5)

In [ ]:
def fix_thm(df):
    thm = [f"thm_{i}" for i in range(1,6)]
    return df.groupby("sequence_id", group_keys=False)[thm].apply(
        lambda g: g.interpolate(limit_direction="both").ffill().bfill()
    )

### 3. ToF – 5 cảm biến × 64 pixel (tof_i_vj)

In [ ]:
def fix_tof_and_reduce(df):
    # 1) sentinel -> NaN + flag
    tof_pix = [c for c in df.columns if c.startswith("tof_") and "_v" in c]
    df[tof_pix] = df[tof_pix].replace(-1, np.nan)
    # optional flags per sensor
    for i in range(1,6):
        cols = [f"tof_{i}_v{p}" for p in range(64) if f"tof_{i}_v{p}" in df]
        df[f"tof_{i}_missing_frac"] = df[cols].isna().mean(axis=1)
        df[f"tof_{i}_mean"] = df[cols].mean(axis=1)  # dimension reduction
    # 2) interpolate the 5 means
    mean_cols = [f"tof_{i}_mean" for i in range(1,6)]
    df[mean_cols] = df.groupby("sequence_id", group_keys=False)[mean_cols].apply(
        lambda g: g.interpolate(limit_direction="both").ffill().bfill()
    )
    # 3) drop heavy pixel columns to save RAM
    df.drop(columns=tof_pix, inplace=True)
    return df

### 4. Pha/nhãn & metadata

In [ ]:
from sklearn.preprocessing import LabelEncoder
le_gesture = LabelEncoder().fit(train_df["gesture"].astype(str))
train_df["gesture_id"] = le_gesture.transform(train_df["gesture"].astype(str))

### 5. Pipeline tiền xử lý

In [ ]:
def minimal_preprocess(df):
    df = df.copy()
    # 1) IMU
    df = fix_imu(df)
    # 2) Thermopile
    thm_fixed = fix_thm(df)  # returns only thm cols
    df[thm_fixed.columns] = thm_fixed
    # 3) ToF -> means + flags, drop pixels
    df = fix_tof_and_reduce(df)
    # 4) Feature cơ bản (ví dụ):
    df["acc_mag"] = np.sqrt(df["acc_x"]**2 + df["acc_y"]**2 + df["acc_z"]**2)
    df["rot_angle"] = 2*np.arccos(np.clip(df["rot_w"], -1, 1))
    return df

# Notebook 2 – Deep Learning: Mô hình chuỗi

**Mục tiêu:**  
Chứng minh mô hình deep learning vượt baseline cổ điển bằng cách huấn luyện các mô hình chuỗi cho **IMU & Thermopile** (1D) và **2D CNN** cho **ToF grid**, sau đó so sánh theo từng modality.

**Bao gồm:**
- **1D CNN / LSTM / GRU / BiLSTM** cho IMU & Thermopile.  
- **2D CNN** cho ToF grid (xử lý từng khung rồi gộp theo thời gian).  
- **Hybrid CNN+LSTM** cho chuỗi 1D.  
- Chia **train/val/test**, **EarlyStopping**, **LR scheduler**.  
- So sánh: **IMU-only / ToF-only / Multimodal (concat đơn giản)**.  
- **Regularization test**: Dropout, LayerNorm để kiểm soát overfit.  
- **Explainability**: saliency map (1D/2D) và/hoặc attention weights để minh hoạ cảm biến/thời điểm quan trọng.

👉 **Vai trò:** Benchmark DL để đối chiếu và chứng minh vượt baseline từ Notebook 1.


In [ ]:

import os, math, json, time, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    from torch.optim import Adam
    from sklearn.metrics import f1_score, classification_report, confusion_matrix
    HAS_TORCH = True
except Exception as e:
    HAS_TORCH = False
    print('Chưa có PyTorch. Cài torch để chạy các ô huấn luyện. Lỗi:', e)

DEVICE = 'cuda' if HAS_TORCH and torch.cuda.is_available() else 'cpu'
SEED = 42
random.seed(SEED); np.random.seed(SEED)
if HAS_TORCH:
    torch.manual_seed(SEED)
    if DEVICE=='cuda': torch.cuda.manual_seed_all(SEED)
print({'torch': HAS_TORCH, 'device': DEVICE})


## 1) Cấu hình thí nghiệm

- Có thể dùng dữ liệu **thật** (file `.npz`) hoặc để trống để sinh **synthetic dataset** chạy thử.
- File `.npz` kỳ vọng chứa các mảng:
  - `imu (N,T,IMU_CH)`
  - `thermo (N,T,THERM_CH)`
  - `tof (N,T,H,W)`
  - `label (N,)`

## 2) Sinh dữ liệu / Tải dữ liệu

- Dữ liệu gồm ba nhánh: **IMU**, **Thermopile** (chuỗi 1D theo thời gian), **ToF** (lưới 2D theo thời gian).
- Nếu không có file thật, mã sẽ tạo **synthetic dataset** với pattern khác nhau giữa các lớp để demo pipeline.


In [ ]:
# %% [config]
CONFIG = {
    'DATA_PATH': '',  # ví dụ: '/mnt/data/your_dataset.npz'

    # Synthetic dataset nếu không có dữ liệu thật
    'N_SAMPLES': 900,
    'N_CLASSES': 6,
    'T': 64,
    'IMU_CH': 6,
    'THERM_CH': 8,
    'TOF_H': 8, 'TOF_W': 8,

    # Huấn luyện
    'BATCH_SIZE': 64,
    'MAX_EPOCHS': 10,
    'LR': 1e-3,
    'WEIGHT_DECAY': 1e-4,
    'EARLY_STOP_PATIENCE': 5,
    'VAL_SPLIT': 0.15,
    'TEST_SPLIT': 0.15,

    # Regularization
    'DROPOUT': 0.2,
    'USE_LAYER_NORM': True,

    # Danh sách thí nghiệm cần chạy
    'RUN_SET': [
        'IMU_CNN1D', 'IMU_LSTM', 'IMU_GRU', 'IMU_BiLSTM',
        'TOF_CNN2D',
        'MULTI_CONCAT_CNN1D+CNN2D'
    ]
}
CONFIG


In [ ]:
# %% [data]
from typing import Dict

def make_synthetic(cfg: Dict):
    N, C, T = cfg['N_SAMPLES'], cfg['N_CLASSES'], cfg['T']
    IMU_CH, THERM_CH = cfg['IMU_CH'], cfg['THERM_CH']
    H, W = cfg['TOF_H'], cfg['TOF_W']
    y = np.random.randint(0, C, size=N)
    imu = np.random.randn(N, T, IMU_CH).astype('float32')
    thermo = (0.3*np.random.randn(N, T, THERM_CH) + 0.1).astype('float32')
    tof = (np.random.randn(N, T, H, W)*0.2 + 2.5).astype('float32')
    # pattern theo lớp
    for cls in range(C):
        idx = y==cls
        imu[idx] += (cls - C/2) * 0.15
        thermo[idx] += (C/2 - cls) * 0.1
        rr, cc = np.mgrid[:H, :W]
        center_r, center_c = (cls % H), (cls % W)
        mask = np.exp(-((rr-center_r)**2 + (cc-center_c)**2)/(2*(H/4)**2)).astype('float32')
        tof[idx] += mask[None, None, :, :]
    return imu, thermo, tof, y.astype('int64')

DATA_PATH = CONFIG['DATA_PATH']
if DATA_PATH and os.path.exists(DATA_PATH):
    data = np.load(DATA_PATH)
    imu = data['imu'].astype('float32')
    thermo = data['thermo'].astype('float32')
    tof = data['tof'].astype('float32')
    y = data['label'].astype('int64')
    print('Loaded real dataset:', DATA_PATH)
else:
    imu, thermo, tof, y = make_synthetic(CONFIG)
    print('Using synthetic dataset')

N, T, IMU_CH = imu.shape
THERM_CH = thermo.shape[-1]
H, W = tof.shape[2], tof.shape[3]
N_CLASSES = int(y.max())+1
print({'N': N, 'T': T, 'IMU_CH': IMU_CH, 'THERM_CH': THERM_CH, 'ToF': (H, W), 'Classes': N_CLASSES})


## 3) Chia tập train / val / test

- Trộn chỉ số mẫu rồi cắt theo tỷ lệ trong `CONFIG`:
  - `TEST_SPLIT` → tập test
  - `VAL_SPLIT` → tập validation
  - phần còn lại → tập train

## 4) Dataset & Dataloader (PyTorch)

- `SeqDataset` trả về 1 mẫu gồm: `imu (T, IMU_CH)`, `thermo (T, THERM_CH)`, `tof (T, H, W)`, `y (int)`.
- `DataLoader`:
  - `shuffle=True` cho **train**
  - batch size lấy từ `CONFIG['BATCH_SIZE']`


In [ ]:
# %% [split]
idx = np.arange(N)
np.random.shuffle(idx)

TEST = int(CONFIG['TEST_SPLIT'] * N)
VAL  = int(CONFIG['VAL_SPLIT']  * N)

test_idx  = idx[:TEST]
val_idx   = idx[TEST:TEST+VAL]
train_idx = idx[TEST+VAL:]

def take(a, ind):
    return a[ind]

split = {
    'train': {
        'imu': take(imu, train_idx),
        'thermo': take(thermo, train_idx),
        'tof': take(tof, train_idx),
        'y': take(y, train_idx),
    },
    'val': {
        'imu': take(imu, val_idx),
        'thermo': take(thermo, val_idx),
        'tof': take(tof, val_idx),
        'y': take(y, val_idx),
    },
    'test': {
        'imu': take(imu, test_idx),
        'thermo': take(thermo, test_idx),
        'tof': take(tof, test_idx),
        'y': take(y, test_idx),
    },
}

for k in split:
    print(k, {kk: v.shape for kk, v in split[k].items()})


In [ ]:
# %% [dataset]
if HAS_TORCH:
    class SeqDataset(Dataset):
        def __init__(self, pack):
            self.imu = pack['imu']        # (N, T, IMU_CH)
            self.thermo = pack['thermo']  # (N, T, THERM_CH)
            self.tof = pack['tof']        # (N, T, H, W)
            self.y = pack['y']            # (N,)
        def __len__(self):
            return self.y.shape[0]
        def __getitem__(self, i):
            return {
                'imu': torch.from_numpy(self.imu[i]),
                'thermo': torch.from_numpy(self.thermo[i]),
                'tof': torch.from_numpy(self.tof[i]),
                'y': torch.tensor(int(self.y[i])),
            }

    loaders = {}
    for name in ['train', 'val', 'test']:
        ds = SeqDataset(split[name])
        loaders[name] = DataLoader(
            ds,
            batch_size=CONFIG['BATCH_SIZE'],
            shuffle=(name == 'train')
        )
else:
    print('Chưa có PyTorch – bỏ qua bước Dataset/Dataloader')


## 5) Kiến trúc mô hình

### 5.1 Encoder 1D (cho IMU & Thermopile)
- **CNN1DEncoder**: 2 conv + (tuỳ chọn) **LayerNorm**, gộp thời gian bằng **AdaptiveAvgPool1d**.
- **LSTMEncoder / GRUEncoder**: lấy **timestep cuối** làm đại diện.
- **BiLSTM**: LSTM 2 chiều (bidir=True).
- **CNN1D+LSTM** (hybrid): conv 1D → (LN) → LSTM.

### 5.2 Encoder 2D (cho ToF)
- **CNN2DEncoder**: xử lý từng frame (T × H × W) → pooling không gian → **trung bình theo thời gian**.

### 5.3 Đầu phân loại
- **ClassifierHead**: Linear → ReLU → Linear (kèm Dropout).
- Metric chính sẽ là **macro-F1** ở phần huấn luyện.

### 5.4 Mô hình đầy đủ theo modality
- **ModelIMU(kind)**: chọn encoder 1D theo `kind ∈ {CNN1D, LSTM, GRU, BiLSTM, CNN1D+LSTM}`.
- **ModelToF**: dùng CNN2DEncoder.
- **ModelMultimodalConcat**: mã hoá **IMU**, **Thermo**, *(tuỳ chọn)* **ToF**, sau đó **concat** đặc trưng và phân loại.


In [ ]:
# %% [models]
if HAS_TORCH:
    # ---- LayerNorm cho tensor 1D (B, C, T) ----
    class LayerNorm1D(nn.Module):
        def __init__(self, n_channels):
            super().__init__()
            self.ln = nn.LayerNorm(n_channels)
        def forward(self, x):
            # x: [B, C, T] -> [B, T, C] -> LN -> [B, C, T]
            x = x.permute(0, 2, 1)
            x = self.ln(x)
            return x.permute(0, 2, 1)

    # ---- CNN1D Encoder ----
    class CNN1DEncoder(nn.Module):
        def __init__(self, in_ch, hidden=64, dropout=0.0, use_ln=True):
            super().__init__()
            self.conv1 = nn.Conv1d(in_ch, hidden, kernel_size=5, padding=2)
            self.conv2 = nn.Conv1d(hidden, hidden, kernel_size=3, padding=1)
            self.ln = LayerNorm1D(hidden) if use_ln else nn.Identity()
            self.pool = nn.AdaptiveAvgPool1d(1)
            self.drop = nn.Dropout(dropout)
        def forward(self, x):
            # x: [B, T, C] -> [B, C, T]
            x = x.transpose(1, 2)
            x = F.relu(self.conv1(x))
            x = F.relu(self.conv2(x))
            x = self.ln(x)
            x = self.pool(x).squeeze(-1)  # [B, hidden]
            return self.drop(x)

    # ---- LSTM/GRU Encoders ----
    class LSTMEncoder(nn.Module):
        def __init__(self, in_ch, hidden=64, bidir=False, dropout=0.0):
            super().__init__()
            self.rnn = nn.LSTM(
                input_size=in_ch, hidden_size=hidden,
                batch_first=True, bidirectional=bidir,
                dropout=dropout if bidir else 0.0
            )
            self.out_ch = hidden * (2 if bidir else 1)
        def forward(self, x):
            out, _ = self.rnn(x)         # [B, T, H*dir]
            return out[:, -1, :]         # lấy timestep cuối

    class GRUEncoder(nn.Module):
        def __init__(self, in_ch, hidden=64, bidir=False, dropout=0.0):
            super().__init__()
            self.rnn = nn.GRU(
                input_size=in_ch, hidden_size=hidden,
                batch_first=True, bidirectional=bidir,
                dropout=dropout if bidir else 0.0
            )
            self.out_ch = hidden * (2 if bidir else 1)
        def forward(self, x):
            out, _ = self.rnn(x)
            return out[:, -1, :]

    # ---- Attention pooling (tuỳ chọn, dùng ở phần Explainability) ----
    class AttentionPooling1D(nn.Module):
        def __init__(self, in_ch):
            super().__init__()
            self.w = nn.Linear(in_ch, 1)
        def forward(self, x):
            # x: [B, T, C]
            alpha = torch.softmax(self.w(x).squeeze(-1), dim=1)  # [B, T]
            ctx = torch.bmm(alpha.unsqueeze(1), x).squeeze(1)    # [B, C]
            return ctx, alpha

    # ---- CNN1D + LSTM (Hybrid) ----
    class CNN1D_LSTM_Encoder(nn.Module):
        def __init__(self, in_ch, hidden=64, dropout=0.0, use_ln=True):
            super().__init__()
            self.conv = nn.Conv1d(in_ch, hidden, kernel_size=5, padding=2)
            self.ln = LayerNorm1D(hidden) if use_ln else nn.Identity()
            self.rnn = nn.LSTM(hidden, hidden, batch_first=True)
            self.drop = nn.Dropout(dropout)
        def forward(self, x):
            h = x.transpose(1, 2)               # [B, C, T]
            h = F.relu(self.conv(h))            # [B, H, T]
            h = self.ln(h).transpose(1, 2)      # [B, T, H]
            out, _ = self.rnn(h)                # [B, T, H]
            return self.drop(out[:, -1, :])     # [B, H]

    # ---- CNN2D Encoder cho ToF ----
    class CNN2DEncoder(nn.Module):
        def __init__(self, in_ch=1, hidden=64, dropout=0.0):
            super().__init__()
            self.conv1 = nn.Conv2d(in_ch, 32, kernel_size=3, padding=1)
            self.conv2 = nn.Conv2d(32, hidden, kernel_size=3, padding=1)
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            self.drop = nn.Dropout(dropout)
        def forward(self, x):
            # x: [B, T, 1, H, W] -> xử lý từng frame rồi trung bình theo thời gian
            B, T, C, H, W = x.shape
            h = x.reshape(B*T, C, H, W)
            h = F.relu(self.conv1(h))
            h = F.relu(self.conv2(h))
            h = self.pool(h).squeeze(-1).squeeze(-1)   # [B*T, hidden]
            h = h.view(B, T, -1).mean(dim=1)           # [B, hidden]
            return self.drop(h)

    # ---- Đầu phân loại ----
    class ClassifierHead(nn.Module):
        def __init__(self, in_ch, n_classes, dropout=0.0):
            super().__init__()
            hid = in_ch//2 if in_ch >= 64 else max(32, in_ch)
            self.net = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(in_ch, hid), nn.ReLU(),
                nn.Linear(hid, n_classes)
            )
        def forward(self, x):
            return self.net(x)

    # ---- Mô hình hoàn chỉnh theo modality ----
    class ModelIMU(nn.Module):
        def __init__(self, kind, in_ch, n_classes, cfg):
            super().__init__()
            dp, ln = cfg['DROPOUT'], cfg['USE_LAYER_NORM']
            if kind == 'CNN1D':
                self.enc = CNN1DEncoder(in_ch, dropout=dp, use_ln=ln); enc_out = 64
            elif kind == 'LSTM':
                self.enc = LSTMEncoder(in_ch, hidden=64, bidir=False, dropout=dp); enc_out = self.enc.out_ch
            elif kind == 'GRU':
                self.enc = GRUEncoder(in_ch, hidden=64, bidir=False, dropout=dp); enc_out = self.enc.out_ch
            elif kind == 'BiLSTM':
                self.enc = LSTMEncoder(in_ch, hidden=64, bidir=True, dropout=dp); enc_out = self.enc.out_ch
            elif kind == 'CNN1D+LSTM':
                self.enc = CNN1D_LSTM_Encoder(in_ch, hidden=64, dropout=dp, use_ln=ln); enc_out = 64
            else:
                raise ValueError('Unknown kind for ModelIMU')
            self.head = ClassifierHead(enc_out, n_classes, dropout=dp)
        def forward(self, imu):
            return self.head(self.enc(imu))

    class ModelToF(nn.Module):
        def __init__(self, n_classes, cfg):
            super().__init__()
            dp = cfg['DROPOUT']
            self.enc = CNN2DEncoder(in_ch=1, hidden=64, dropout=dp)
            self.head = ClassifierHead(64, n_classes, dropout=dp)
        def forward(self, tof):
            return self.head(self.enc(tof))

    class ModelMultimodalConcat(nn.Module):
        def __init__(self, imu_kind, thermo_kind, use_tof, in_imu, in_thermo, n_classes, cfg):
            super().__init__()
            dp = cfg['DROPOUT']
            # encoders
            self.enc_imu = ModelIMU(imu_kind, in_imu, n_classes, cfg).enc
            self.enc_thermo = ModelIMU(thermo_kind, in_thermo, n_classes, cfg).enc
            self.use_tof = use_tof
            if use_tof:
                self.enc_tof = CNN2DEncoder(in_ch=1, hidden=64, dropout=dp)
            # suy ra kích thước đặc trưng
            d_imu = 64 if imu_kind in ['CNN1D','CNN1D+LSTM'] else (128 if imu_kind=='BiLSTM' else 64)
            d_th  = 64 if thermo_kind in ['CNN1D','CNN1D+LSTM'] else (128 if thermo_kind=='BiLSTM' else 64)
            d_tof = 64 if use_tof else 0
            self.head = ClassifierHead(d_imu + d_th + d_tof, n_classes, dropout=dp)
        def forward(self, imu, thermo, tof):
            feats = [self.enc_imu(imu), self.enc_thermo(thermo)]
            if self.use_tof:
                feats.append(self.enc_tof(tof))
            return self.head(torch.cat(feats, dim=-1))
else:
    print('Chưa có torch – bỏ qua định nghĩa mô hình')


## 6) Tiện ích huấn luyện

- **Metric chính**: `macro-F1` (cân bằng giữa các lớp).
- **EarlyStopping**: dừng sớm khi `val_loss` không cải thiện sau `patience` epoch.
- **LR Scheduler**: `ReduceLROnPlateau` (giảm LR khi `val_loss` không giảm).
- Vòng lặp:
  - `run_epoch(...)` cho train/val (tự phát hiện mô hình IMU/ToF/Multimodal).
  - `fit_model(...)` huấn luyện đầy đủ, lưu lịch sử loss/F1/LR theo epoch, tải lại **trạng thái tốt nhất** (best `val_loss`).


In [ ]:
# %% [train_utils]
if HAS_TORCH:
    from typing import Dict, Tuple, List

    # ---- Macro-F1 tiện ích ----
    def macro_f1(y_true, y_pred):
        return f1_score(y_true, y_pred, average='macro')

    # ---- EarlyStopping đơn giản ----
    class EarlyStopping:
        def __init__(self, patience: int = 5, min_delta: float = 0.0):
            self.patience = patience
            self.min_delta = min_delta
            self.best = None
            self.count = 0
            self.stop = False

        def step(self, value: float):
            # value: val_loss hiện tại
            if self.best is None or value < self.best - self.min_delta:
                self.best = value
                self.count = 0
            else:
                self.count += 1
                if self.count >= self.patience:
                    self.stop = True

    # ---- Chạy một epoch (train hoặc val) ----
    def run_epoch(model, loader, criterion, optimizer=None):
        """
        Trả về: (mean_loss, macro_f1)
        """
        is_train = optimizer is not None
        model.train() if is_train else model.eval()

        total_loss = 0.0
        ys, yhs = [], []

        for b in loader:
            imu_b = b['imu'].to(DEVICE).float()
            th_b  = b['thermo'].to(DEVICE).float()
            tof_b = b['tof'].to(DEVICE).float().unsqueeze(2)  # [B, T, 1, H, W]
            y_b   = b['y'].to(DEVICE)

            if is_train:
                optimizer.zero_grad()

            # Chọn đường đi forward theo loại mô hình
            if isinstance(model, ModelIMU):
                logits = model(imu_b)                       # IMU-only
            elif isinstance(model, ModelToF):
                 logits = model(tof_b)                       # ToF-only
            else:
                 logits = model(imu_b, th_b, tof_b)

            loss = criterion(logits, y_b)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * y_b.size(0)
            ys.append(y_b.detach().cpu().numpy())
            yhs.append(logits.detach().cpu().argmax(dim=1).numpy())

        ys = np.concatenate(ys)
        yhs = np.concatenate(yhs)
        return total_loss / len(loader.dataset), macro_f1(ys, yhs)

    # ---- Huấn luyện đầy đủ với EarlyStopping + LR scheduler ----
    def fit_model(model, loaders: Dict[str, torch.utils.data.DataLoader], cfg: Dict, tag: str = 'model'):
        model = model.to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = Adam(model.parameters(), lr=cfg['LR'], weight_decay=cfg['WEIGHT_DECAY'])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=2
        )
        early = EarlyStopping(patience=cfg['EARLY_STOP_PATIENCE'])

        hist = {'epoch': [], 'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': [], 'lr': []}
        best_state = None
        best_val = float('inf')

        for ep in range(cfg['MAX_EPOCHS']):
            tr_loss, tr_f1 = run_epoch(model, loaders['train'], criterion, optimizer)
            vl_loss, vl_f1 = run_epoch(model, loaders['val'],   criterion, optimizer=None)

            scheduler.step(vl_loss)
            lr_now = optimizer.param_groups[0]['lr']

            hist['epoch'].append(ep + 1)
            hist['train_loss'].append(tr_loss)
            hist['val_loss'].append(vl_loss)
            hist['train_f1'].append(tr_f1)
            hist['val_f1'].append(vl_f1)
            hist['lr'].append(lr_now)

            # Lưu trạng thái tốt nhất theo val_loss
            if vl_loss < best_val:
                best_val = vl_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

            # In log ngắn gọn
            print(f"[EP {ep+1:02d}] "
                  f"tr_loss={tr_loss:.4f} val_loss={vl_loss:.4f} "
                  f"tr_f1={tr_f1:.4f} val_f1={vl_f1:.4f} lr={lr_now:.2e}")

            # Early stopping
            early.step(vl_loss)
            if early.stop:
                print("Dừng sớm (EarlyStopping).")
                break

        # Tải lại trọng số tốt nhất
        if best_state is not None:
            model.load_state_dict(best_state)

        return model, hist
else:
    print('Chưa có torch – bỏ qua utilities huấn luyện')


## 7) Chạy thí nghiệm & so sánh theo modality

- Chạy lần lượt các cấu hình trong `CONFIG['RUN_SET']`:
  - **IMU-only**: `IMU_CNN1D`, `IMU_LSTM`, `IMU_GRU`, `IMU_BiLSTM`
  - **ToF-only**: `TOF_CNN2D`
  - **Multimodal**: `MULTI_CONCAT_CNN1D+CNN2D` (IMU+Thermo+(ToF))

- Mỗi mô hình:
  - Huấn luyện với **EarlyStopping** + **ReduceLROnPlateau**.
  - Ghi lại **macro-F1** trên train/val/test.
  - Tổng hợp bảng kết quả để dễ so sánh mô hình & modality.

> Lưu ý:
> - Bạn có thể chỉnh danh sách `RUN_SET` để chạy ít mô hình hơn cho nhanh.
> - Nếu môi trường chưa cài `torch`, phần này sẽ bị bỏ qua.


In [ ]:
# %% [experiments]
import pandas as pd
from pathlib import Path

results = []
histories = {}

if HAS_TORCH:
    @torch.no_grad()
    def evaluate_test(model, loader):
        model.eval()
        ys, yhs = [], []
        for b in loader:
            imu_b = b['imu'].to(DEVICE).float()
            th_b  = b['thermo'].to(DEVICE).float()
            tof_b = b['tof'].to(DEVICE).float().unsqueeze(2)  # [B, T, 1, H, W]
            y_b   = b['y'].to(DEVICE)

            if isinstance(model, ModelIMU):
                logits = model(imu_b)
            elif isinstance(model, ModelToF):
                logits = model(tof_b)
            else:
                logits = model(imu_b, th_b, tof_b)

            ys.append(y_b.cpu().numpy())
            yhs.append(logits.argmax(dim=1).cpu().numpy())

        ys = np.concatenate(ys)
        yhs = np.concatenate(yhs)
        return f1_score(ys, yhs, average='macro')

    RUN_SET = CONFIG['RUN_SET']
    for tag in RUN_SET:
        if tag.startswith('IMU_'):
            kind = tag.split('_', 1)[1]
            model = ModelIMU(kind, in_ch=IMU_CH, n_classes=N_CLASSES, cfg=CONFIG)
            modality = 'IMU'
        elif tag == 'TOF_CNN2D':
            model = ModelToF(n_classes=N_CLASSES, cfg=CONFIG)
            modality = 'ToF'
        elif tag.startswith('MULTI_CONCAT'):
            model = ModelMultimodalConcat(
                imu_kind='CNN1D',
                thermo_kind='CNN1D',
                use_tof=True,
                in_imu=IMU_CH, in_thermo=THERM_CH,
                n_classes=N_CLASSES, cfg=CONFIG
            )
            modality = 'Multimodal'
        else:
            print(f'Bỏ qua tag không hỗ trợ: {tag}')
            continue

        model, hist = fit_model(model, loaders, CONFIG, tag)
        hist_df = pd.DataFrame(hist)
        histories[tag] = hist_df

        tr_f1 = hist_df['train_f1'].iloc[-1]
        val_f1 = hist_df['val_f1'].iloc[-1]
        te_f1 = evaluate_test(model, loaders['test'])

        results.append({
            'exp': tag,
            'modality': modality,
            'train_f1': tr_f1,
            'val_f1': val_f1,
            'test_f1': te_f1
        })
        print(f"==> {tag}: test macro-F1 = {te_f1:.4f}")

    res_df = pd.DataFrame(results).sort_values('test_f1', ascending=False)
    display(res_df)

    # Lưu bảng kết quả (tùy chọn)
    out_dir = Path('/mnt/data')
    out_dir.mkdir(parents=True, exist_ok=True)
    res_path = out_dir / 'dl_results.csv'
    res_df.to_csv(res_path, index=False)
    print('Đã lưu bảng kết quả tại:', res_path)
else:
    print('Chưa có PyTorch – bỏ qua phần chạy thí nghiệm')


## 8) Kiểm soát overfit: Dropout & LayerNorm

- Mục tiêu: so sánh đường cong **train/val loss** và **macro-F1** khi:
  1. Không dùng dropout, không dùng layer norm.
  2. Có dropout (0.5) + layer norm.

- Thí nghiệm nhanh với backbone `IMU_CNN1D`.
- Kỳ vọng: khi có Dropout + LayerNorm, mô hình ít bị overfit hơn, đường cong train–val sát nhau hơn.


In [ ]:
# %% [regularization]
if HAS_TORCH:
    def quick_run_imu_cnn1d(dropout, use_ln):
        cfg2 = CONFIG.copy()
        cfg2['DROPOUT'] = dropout
        cfg2['USE_LAYER_NORM'] = use_ln
        model = ModelIMU('CNN1D', in_ch=IMU_CH, n_classes=N_CLASSES, cfg=cfg2)
        model, hist = fit_model(model, loaders, cfg2, tag=f'IMU_CNN1D_dp{dropout}_ln{use_ln}')
        return pd.DataFrame(hist)

    # Chạy 2 cấu hình: không regularization vs có regularization
    h1 = quick_run_imu_cnn1d(dropout=0.0, use_ln=False)
    h2 = quick_run_imu_cnn1d(dropout=0.5, use_ln=True)

    # Vẽ loss curves
    plt.figure()
    plt.plot(h1['epoch'], h1['train_loss'], label='train_loss no-reg')
    plt.plot(h1['epoch'], h1['val_loss'], label='val_loss no-reg')
    plt.plot(h2['epoch'], h2['train_loss'], label='train_loss reg')
    plt.plot(h2['epoch'], h2['val_loss'], label='val_loss reg')
    plt.title('Regularization: Loss Curves')
    plt.legend(); plt.tight_layout(); plt.show()

    # Vẽ macro-F1 curves
    plt.figure()
    plt.plot(h1['epoch'], h1['train_f1'], label='train_f1 no-reg')
    plt.plot(h1['epoch'], h1['val_f1'], label='val_f1 no-reg')
    plt.plot(h2['epoch'], h2['train_f1'], label='train_f1 reg')
    plt.plot(h2['epoch'], h2['val_f1'], label='val_f1 reg')
    plt.title('Regularization: Macro-F1 Curves')
    plt.legend(); plt.tight_layout(); plt.show()
else:
    print('Chưa có PyTorch – bỏ qua kiểm tra regularization')


## 9) Giải thích mô hình (Explainability)

### 9.1 Saliency map
- **IMU (1D)**: tính gradient của logit theo input → heatmap `channels × time`.
- **ToF (2D)**: tính gradient theo input ảnh, lấy trung bình theo thời gian → heatmap không gian.

### 9.2 Attention weights
- Xây một encoder IMU có AttentionPooling1D.
- Huấn luyện nhanh vài epoch.
- Vẽ biểu đồ attention theo thời gian để minh hoạ timestep quan trọng.

> Lưu ý: phần này mang tính minh hoạ trực quan, không bắt buộc phải huấn luyện kỹ.


In [ ]:
# %% [explainability]
if HAS_TORCH:
    # ---- Mô hình nhỏ với Attention ----
    class IMUWithAttention(nn.Module):
        def __init__(self, in_ch, n_classes, cfg):
            super().__init__()
            self.conv = nn.Conv1d(in_ch, 64, kernel_size=5, padding=2)
            self.proj = nn.Linear(64, 64)
            self.attn = AttentionPooling1D(64)
            self.head = ClassifierHead(64, n_classes, dropout=cfg['DROPOUT'])
        def forward(self, x):
            h = x.transpose(1, 2)
            h = F.relu(self.conv(h))        # [B, 64, T]
            h = h.transpose(1, 2)           # [B, T, 64]
            h2 = self.proj(h)
            ctx, alpha = self.attn(h2)
            return self.head(ctx), alpha

    # Saliency 1D cho IMU
    batch = next(iter(loaders['val']))
    x = batch['imu'].to(DEVICE).float()
    x.requires_grad_(True)
    model_plain = ModelIMU('CNN1D', in_ch=IMU_CH, n_classes=N_CLASSES, cfg=CONFIG).to(DEVICE)
    model_plain.eval()
    logits = model_plain(x)
    cls = logits.argmax(dim=1)
    sel = logits[range(x.size(0)), cls].sum()
    sel.backward()
    sal = x.grad.detach().abs().mean(dim=0).cpu().numpy()  # [T, C]
    plt.figure()
    plt.imshow(sal.T, aspect='auto')
    plt.title('IMU Saliency (channels × time)')
    plt.xlabel('time'); plt.ylabel('channels')
    plt.tight_layout(); plt.show()

    # Saliency 2D cho ToF
    x2 = batch['tof'].to(DEVICE).float().unsqueeze(2)
    x2.requires_grad_(True)
    model_tof = ModelToF(n_classes=N_CLASSES, cfg=CONFIG).to(DEVICE)
    model_tof.eval()
    logits2 = model_tof(x2)
    cls2 = logits2.argmax(dim=1)
    sel2 = logits2[range(x2.size(0)), cls2].sum()
    sel2.backward()
    grad = x2.grad.detach().abs().mean(dim=1).squeeze(1).cpu().numpy()  # [B, H, W]
    plt.figure()
    plt.imshow(grad[0], interpolation='nearest')
    plt.title('ToF Saliency (spatial) - sample 0')
    plt.tight_layout(); plt.show()

    # Attention weights minh hoạ
    model_attn = IMUWithAttention(IMU_CH, N_CLASSES, CONFIG).to(DEVICE)
    crit = nn.CrossEntropyLoss()
    opt = Adam(model_attn.parameters(), lr=CONFIG['LR'])
    for _ in range(2):  # train nhanh 2 epoch demo
        model_attn.train()
        for b in loaders['train']:
            x_train = b['imu'].to(DEVICE).float()
            y_train = b['y'].to(DEVICE)
            opt.zero_grad()
            logits, _ = model_attn(x_train)
            loss = crit(logits, y_train)
            loss.backward()
            opt.step()

    model_attn.eval()
    with torch.no_grad():
        _, alpha = model_attn(x)
    a = alpha[0].detach().cpu().numpy()
    plt.figure()
    plt.plot(np.arange(a.shape[0]), a)
    plt.title('Attention weights theo thời gian (IMU sample 0)')
    plt.xlabel('time'); plt.ylabel('weight')
    plt.tight_layout(); plt.show()
else:
    print('Chưa có PyTorch – bỏ qua explainability')


## 10) Kết luận & Ghi chú cuối

- Sau khi chạy các thí nghiệm, kết quả **macro-F1** trên tập test sẽ được lưu trong bảng `dl_results.csv` (trong thư mục `/mnt/data`).
- Bảng này có thể ghép so sánh trực tiếp với kết quả **baseline classical ML** ở Notebook 1 để minh chứng:
  - **Deep Learning** (CNN/LSTM/GRU/BiLSTM, Multimodal) thường vượt trội hơn baseline (LogReg, RF, XGB...).
- **Regularization** (Dropout, LayerNorm) giúp mô hình bớt overfit, đặc biệt trên dữ liệu nhỏ.
- **Explainability** (saliency map, attention) cho phép hiểu rõ:
  - Cảm biến nào (channel) đóng vai trò quan trọng.
  - Thời điểm nào trong chuỗi tín hiệu có ảnh hưởng mạnh đến dự đoán.

👉 Đây là bước quan trọng để chứng minh tính ưu việt của mô hình deep learning, đồng thời bổ sung thêm insight khoa học khi phân tích dữ liệu đa cảm biến.


In [ ]:
# %% [end]
print("Notebook 2 đã hoàn tất. Kết quả đã được lưu ra /mnt/data/dl_results.csv (nếu chạy đầy đủ).")
print("So sánh bảng kết quả này với baseline ở Notebook 1 để minh chứng deep learning vượt baseline.")